# Task 2.2 – Real-time Violation Visualisation

Polls MongoDB every 5 s and updates two subplots whenever new violations are
written by the streaming application. Run alongside
`data_design_streaming.ipynb` to see live updates.

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `POLL_INTERVAL` | 5 s | Seconds between MongoDB polls |
| `WINDOW` | 20 pts | Rolling window – oldest points are dropped |
| Moving avg window | 5 pts | Same as Week-10 Scenario 4 |
| Spike threshold | 1.5 × mean | Flags sudden violation bursts |
| Percentile level | 90th | Dynamically computed on current window |


## Setup

In [ ]:
from time import sleep
from datetime import datetime
import statistics

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from pymongo import MongoClient

# needed for real-time interactive display in Jupyter Notebook
%matplotlib notebook

HOST_IP       = 'host.docker.internal'
POLL_INTERVAL = 5   # seconds between MongoDB polls
WINDOW        = 20  # rolling window size

# read speed limits from camera.csv – no hardcoding
_cam = pd.read_csv('../data/camera.csv')
SPEED_LIMITS = _cam.set_index('camera_id')['speed_limit'].to_dict()
# unique limit values for the axhline reference lines
_unique_limits = sorted(set(SPEED_LIMITS.values()))
print('Speed limits loaded:', SPEED_LIMITS)


## How it works

MongoDB is queried with `$sort + $skip: seen_count` so no timestamp-type
comparison is needed. Plotting starts as soon as the first data point arrives
(`len(x) > 1`), so the chart is visible from the very first poll.

```
seen_count = 0
while True:
    docs = query_new_violations(collection, seen_count)   # $skip approach
    if docs:
        seen_count += len(docs)
        x.append(latest_event_timestamp)
        if len(x) > 1:          # draw from the first data point
            redraw both subplots
            fig.canvas.draw()   # Week-10 style update
            x.pop(0)            # sliding window
    sleep(POLL_INTERVAL)
```


## Task 2.2.1 & 2.2.2 – Visualisation

In [ ]:
# ── annotation helpers (based on Week-10 Scenario 2 / 4 class code) ──────────

def annotate_max(x, y, ax=None):
    ymax = max(y)
    xpos = y.index(ymax)
    xmax = x[xpos]
    text = 'Max: Time={}, Value={}'.format(xmax.strftime('%H:%M:%S'), round(ymax, 1))
    if not ax:
        ax = plt.gca()
    offset = max(y) * 0.12 + 0.1
    ax.annotate(text, xy=(xmax, ymax), xytext=(xmax, ymax + offset),
                arrowprops=dict(facecolor='red', shrink=0.05))

def annotate_min(x, y, ax=None):
    ymin = min(y)
    xpos = y.index(ymin)
    xmin = x[xpos]
    text = 'Min: Time={}, Value={}'.format(xmin.strftime('%H:%M:%S'), round(ymin, 1))
    if not ax:
        ax = plt.gca()
    offset = max(y) * 0.12 + 0.1
    ax.annotate(text, xy=(xmin, ymin), xytext=(xmin, ymin + offset),
                arrowprops=dict(facecolor='orange', shrink=0.05))

# dynamically computes and labels the Nth percentile (HD requirement)
def annotate_percentile(x, y, ax, pct=90):
    p_val = float(np.percentile(y, pct))
    ax.axhline(y=p_val, color='purple', linestyle=':', linewidth=1.5,
               label='P{} = {:.1f} km/h'.format(pct, p_val))
    ax.text(x[0], p_val + 0.5, ' P{} = {:.1f}'.format(pct, p_val),
            color='purple', fontsize=8)

# shades windows where count spikes above factor * mean (Distinction)
def shade_spikes(x, y, ax, factor=1.5):
    mean_val = statistics.mean(y)
    if mean_val == 0:
        return
    labeled = False
    for i, yi in enumerate(y):
        if yi > mean_val * factor:
            xs = x[max(0, i - 1)]
            xe = x[min(len(x) - 1, i + 1)]
            lbl = 'Spike Region' if not labeled else ''
            ax.axvspan(xs, xe, alpha=0.15, color='red', label=lbl)
            labeled = True


# ── MongoDB helpers ───────────────────────────────────────────────────────────

def connect_mongo():
    try:
        client = MongoClient(host=HOST_IP, port=27017)
        return client
    except Exception as e:
        print('MongoDB connection failed:', e)
        return None

def to_datetime(ts):
    # handles both Python datetime and ISO string from MongoDB
    if isinstance(ts, datetime):
        return ts.replace(tzinfo=None)
    s = str(ts).replace('T', ' ').split('.')[0]
    try:
        return datetime.strptime(s, '%Y-%m-%d %H:%M:%S')
    except Exception:
        return datetime.now()

def query_new_violations(collection, skip_count):
    # sort by event timestamp, skip records already processed
    # avoids timestamp-type mismatch issues entirely
    pipeline = [
        {'$unwind': '$violations'},
        {'$sort':   {'violations.timestamp_start': 1}},
        {'$project': {
            '_id': 0,
            'violation_type': '$violations.violation_type',
            'speed_reading':  '$violations.speed_reading',
            'ts':             '$violations.timestamp_start',
        }},
        {'$skip': skip_count},
    ]
    return list(collection.aggregate(pipeline))


# ── plot initialisation (same pattern as Week-10 Consumer 3) ─────────────────

def init_plots():
    fig = plt.figure(figsize=(11, 7))
    fig.subplots_adjust(hspace=0.7)

    ax1 = fig.add_subplot(211)
    ax1.set_xlabel('Event Time')
    ax1.set_ylabel('Violation Count')
    ax1.title.set_text('Violation Count vs Event Time')

    ax2 = fig.add_subplot(212)
    ax2.set_xlabel('Event Time')
    ax2.set_ylabel('Speed (km/h)')
    ax2.title.set_text('Speed Pattern vs Event Time')

    fig.suptitle('AWAS Real-time Violation Dashboard')
    fig.show()
    fig.canvas.draw()
    return fig, ax1, ax2


# ── main real-time loop ───────────────────────────────────────────────────────

def visualize_violations():
    client = connect_mongo()
    if client is None:
        return
    collection = client['traffic_monitoring']['violations']

    fig, ax1, ax2 = init_plots()

    # rolling window containers
    x, y_inst, y_avg, y_speed, y_moving = [], [], [], [], []
    seen_count = 0

    try:
        while True:
            docs = query_new_violations(collection, seen_count)

            if docs:
                seen_count += len(docs)
                latest_ts = to_datetime(docs[-1]['ts'])

                inst     = sum(1 for d in docs if d['violation_type'] == 'INSTANTANEOUS')
                avg      = sum(1 for d in docs if d['violation_type'] == 'AVERAGE')
                spds     = [d['speed_reading'] for d in docs if d['speed_reading'] > 0]
                mean_spd = statistics.mean(spds) if spds else 0

                x.append(latest_ts)
                y_inst.append(inst)
                y_avg.append(avg)
                y_speed.append(mean_spd)

                # 5-window moving average – same as Week-10 Scenario 4
                if len(y_speed) > 5:
                    y_moving.append(statistics.mean(y_speed[-5:]))
                else:
                    y_moving.append(statistics.mean(y_speed))

            # [FIX] draw from the first data point so demo shows chart immediately
            if len(x) > 1:

                # ── subplot 1: violation counts ──────────────────
                ax1.clear()
                ax1.plot(x, y_inst, marker='o', linestyle='-',
                         color='crimson',   linewidth=1.8, markersize=5,
                         label='Instantaneous')
                ax1.plot(x, y_avg,  marker='s', linestyle='--',
                         color='steelblue', linewidth=1.8, markersize=5,
                         label='Average Speed')
                ax1.set_xlabel('Event Time')
                ax1.set_ylabel('Violation Count')
                ax1.set_title('Violation Count vs Event Time')
                ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
                ax1.tick_params(axis='x', rotation=30)

                # Task 2.2.2 annotations
                annotate_max(x, y_inst, ax1)       # max callout (Pass)
                annotate_min(x, y_inst, ax1)       # min callout (Credit)
                shade_spikes(x, y_inst, ax1, 1.5)  # spike shading (Distinction)
                # [FIX] legend after shade_spikes so 'Spike Region' appears in it
                ax1.legend(loc='upper right', fontsize=8)

                # ── subplot 2: speed pattern ─────────────────────
                ax2.clear()
                ax2.plot(x, y_speed,  marker='.', linestyle='-',
                         color='salmon', linewidth=0.8, alpha=0.6,
                         label='Mean Speed (per batch)')
                ax2.plot(x, y_moving, marker='^', linestyle='-',
                         color='navy',   linewidth=2.0, markersize=5,
                         label='Moving Avg (5-win)')

                # speed limit reference lines – read from camera.csv, not hardcoded
                _colours = ['red', 'darkorange', 'green']
                for idx, (cam_id, limit) in enumerate(
                        sorted(set((v, k) for k, v in SPEED_LIMITS.items()))):
                    cam_ids = [k for k, v in SPEED_LIMITS.items() if v == cam_id]
                    lbl = 'Limit Cam {} ({} km/h)'.format(
                        '/'.join(str(c) for c in cam_ids), cam_id)
                    ax2.axhline(y=cam_id,
                                color=_colours[idx % len(_colours)],
                                linestyle='-.', linewidth=1.2, alpha=0.7,
                                label=lbl)

                ax2.set_xlabel('Event Time')
                ax2.set_ylabel('Speed (km/h)')
                ax2.set_title('Speed Pattern vs Event Time')
                ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
                ax2.tick_params(axis='x', rotation=30)

                # Task 2.2.2 annotations
                annotate_max(x, y_moving, ax2)            # max callout (Pass)
                annotate_min(x, y_moving, ax2)            # min callout (Credit)
                annotate_percentile(x, y_moving, ax2, 90) # 90th pct (HD)
                ax2.legend(loc='upper right', fontsize=8)

                fig.canvas.draw()

                # slide the window – same as Week-10 consumer pop(0)
                if len(x) > WINDOW:
                    x.pop(0)
                    y_inst.pop(0)
                    y_avg.pop(0)
                    y_speed.pop(0)
                    y_moving.pop(0)

            else:
                print('Waiting for data... ({} records so far)'.format(seen_count))

            sleep(POLL_INTERVAL)

    except KeyboardInterrupt:
        print('Stopped.')
    finally:
        client.close()
        plt.close('all')


if __name__ == '__main__':
    visualize_violations()


## Interesting Points – Operational Significance

**`annotate_max` / `annotate_min`** — shows which minute had the highest and
lowest violation activity in the visible window. Enforcement teams can use the
peak timestamp to pinpoint the most dangerous interval.

**`shade_spikes`** — shades any window where the count exceeds 1.5 x the mean.
A sudden surge may indicate bunched traffic from an upstream incident.

**`annotate_percentile(pct=90)`** — draws a dynamically computed 90th-percentile
line on the speed plot. Vehicles consistently above this threshold are the top
10% speeders, which could justify heavier penalties under a tiered policy.

**Speed limit `axhline`** — reference lines are read from `camera.csv` rather
than hardcoded, so the chart stays correct if camera configurations change.
Any moving-average value above a dashed line indicates a sustained exceedance,
not just a momentary spike.
